In [ ]:
import sys
from pathlib import Path

# Tu dong kiem tra moi truong Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    DRIVE_MY = Path('/content/drive/MyDrive')
else:
    DRIVE_MY = Path('G:/My Drive')

if IN_COLAB:
    try:
        from google.colab import drive
        if not Path('/content/drive/MyDrive').exists():
            drive.mount('/content/drive')
        print('Drive mounted ->', DRIVE_MY)
    except Exception as exc:
        raise RuntimeError('Khong mount duoc Google Drive: %s' % exc)
else:
    print('Local mode: bo qua mount. Kiem tra o G:\\ ...')
    print('  G:\\ ton tai      :', Path('G:/').exists())
    print('  My Drive ton tai :', DRIVE_MY.exists())

Mounted at /content/drive
Drive mounted -> /content/drive/MyDrive


In [ ]:
import collections
names = [i.filename for i in zf.infolist() if '/clips/' in i.filename]
parts = [Path(n).stem.split('__') for n in names]
for k, label in ((1, 'goc quay'), (2, 'cam xuc'), (3, 'muc')):
    print(label, dict(collections.Counter(p[k] for p in parts)))

def show(n):
    raw = io.BytesIO(zf.read(n))
    try:
        z = np.load(raw, allow_pickle=False)
    except Exception:
        raw.seek(0); z = np.load(raw, allow_pickle=True)
    print('\n#', n)
    for k in z.files:
        a = z[k]
        s = (' = %s' % a.tolist()) if a.size <= 12 else ''
        print('   %-28s shape=%-18s dtype=%s%s' % (k, a.shape, a.dtype, s))

show(names[0])
neu = [n for n in names if '__neutral__' in n]
if neu: show(neu[0])
show('W028/statistics/derived.npz')

for f in ('W028/actor_metadata.json', 'W028/statistics/summary.json', 'W028/errors/errors.json'):
    print('\n=====', f); print(zf.read(f).decode('utf-8')[:1500])

print('\n===== stage_b_mead_out/')
for p in sorted(d.parent.iterdir())[:60]:
    print('  ', p.name)

goc quay {'down': 657, 'front': 657, 'left_30': 657, 'right_30': 657}
cam xuc {'angry': 348, 'contempt': 348, 'disgusted': 348, 'fear': 344, 'happy': 360, 'neutral': 160, 'sad': 360, 'surprised': 360}
muc {'L1': 984, 'L2': 836, 'L3': 808}

# W028/clips/W028__down__angry__L1__001.npz
   landmark_raw                 shape=(121, 68, 3)       dtype=float32
   landmark_norm                shape=(121, 68, 3)       dtype=float32
   face_scale                   shape=(121,)             dtype=float32
   pose                         shape=(121, 3)           dtype=float32
   pose_matrix                  shape=(121, 4, 4)        dtype=float32
   blendshape                   shape=(121, 52)          dtype=float32
   confidence                   shape=(121, 68)          dtype=float32
   bbox                         shape=(121, 4)           dtype=float32
   valid_mask                   shape=(121,)             dtype=uint8
   frame_idx                    shape=(121,)             dtype=int32
   timesta

In [ ]:
import os, io, json, zipfile
from pathlib import Path
import numpy as np

def find(name, root='/content/drive/MyDrive', depth=6):
    root = Path(root); base = len(root.parts)
    for dp, dn, fn in os.walk(root):
        if len(Path(dp).parts) - base >= depth:
            dn[:] = []
        if name in fn or name in dn:
            return Path(dp)
    return None

d = find('W028_landmark_package.zip')
print('Thu muc:', d)

for fn, n in (('manifest.json', 3000), ('receipt.json', 1500)):
    print('\n=====', fn)
    print((d / fn).read_text(encoding='utf-8')[:n])

print('\n===== ckpt/')
for p in sorted((d / 'ckpt').iterdir())[:30]:
    print('  %-50s %10.1f MB' % (p.name, p.stat().st_size / 1e6))

print('\n===== W028_landmark_package.zip')
zf = zipfile.ZipFile(d / 'W028_landmark_package.zip')
infos = zf.infolist()
exts = {}
for i in infos:
    e = Path(i.filename).suffix.lower()
    exts[e] = exts.get(e, 0) + 1
print('so file:', len(infos), '| theo duoi:', exts)
for i in infos[:40]:
    line = '  %-70s %9.2f MB' % (i.filename, i.file_size / 1e6)
    if i.filename.endswith('.npy') and i.file_size < 200e6:
        a = np.load(io.BytesIO(zf.read(i.filename)), allow_pickle=False)
        line += '  shape=%s dtype=%s' % (a.shape, a.dtype)
    print(line)

Thu muc: /content/drive/MyDrive/DAT301m/Data_student/Data_emotion_mead_task_b/stage_b_mead_out/W028

===== manifest.json
{
 "actor_id": "W028",
 "status": "COMPLETED",
 "total_clips": 2628,
 "processed_clips": 2628,
 "failed_clips": 0,
 "total_frames": 302568,
 "valid_frames": 302543,
 "expected_clips": 2628,
 "scan_complete": true,
 "pipeline_version": "3.0.0",
 "feature_version": "landmark_v4",
 "clip_schema": "clip_v4",
 "config_hash": "ef2c312a9b83b8d2",
 "test_mode": false,
 "created_at": "2026-10-06T04:25:44"
}

===== receipt.json
{
 "actor_id": "W028",
 "status": "COMPLETED",
 "zip": "W028_landmark_package.zip",
 "sha256": "38d049669ff40e8e90c679f7a5c1514b1a659db393413566e28879bcef59c6e1",
 "n_bytes": 656599442,
 "n_clips": 2628,
 "failed_clips": 0,
 "total_frames": 302568,
 "config_hash": "ef2c312a9b83b8d2",
 "pipeline_version": "3.0.0",
 "feature_version": "landmark_v4",
 "test_mode": false,
 "verified": true,
 "completed_at": "2026-10-06T04:25:44"
}

===== ckpt/
  state.json 

In [ ]:
import tarfile, time, zipfile, collections, subprocess, shutil
from pathlib import Path
import numpy as np

src = Path('/content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD/W028')
tf = tarfile.open(src / 'audio.tar')
mem = [m for m in tf.getmembers() if m.isfile()]
print('audio.tar:', len(mem), 'file |', collections.Counter(Path(m.name).suffix for m in mem))
for m in mem[:8]:
    print('  ', m.name, m.size)

# lay 1 file audio front/neutral roi doi chieu voi landmark cung clip
cand = [m for m in mem if '/front/' in m.name and '/neutral/' in m.name]
print('front/neutral:', len(cand))
if cand:
    m = cand[0]; tf.extract(m, '/tmp/mead_audio'); p = Path('/tmp/mead_audio') / m.name
    try:
        import librosa
        y, sr = librosa.load(str(p), sr=16000)
        print('doc duoc:', p.name, '| thoi luong %.3f s | mel frame (hop 256) ~ %.1f' % (len(y) / sr, len(y) / 256))
    except Exception as e:
        print('Khong doc duoc audio:', repr(e)[:300])
    sent = Path(m.name).stem
    lv = [x for x in m.name.split('/') if x.startswith('level')][0].replace('level_', 'L')
    z = np.load(io.BytesIO(zf.read('W028/clips/W028__front__neutral__%s__%s.npz' % (lv, sent))))
    print('landmark cung clip: %d frame @30fps = %.3f s' % (len(z['landmark_norm']), len(z['landmark_norm']) / 30))

# so clip front / neutral cua tung speaker (chi doc danh sach file trong zip)
root = Path('/content/drive/MyDrive/DAT301m/Data_student/Data_emotion_mead_task_b/stage_b_mead_out')
t0 = time.time()
for sp in sorted(p for p in root.iterdir() if p.is_dir()):
    zp = sp / ('%s_landmark_package.zip' % sp.name)
    if not zp.exists():
        print('%-5s KHONG CO zip' % sp.name); continue
    try:
        nm = [n for n in zipfile.ZipFile(zp).namelist() if '/clips/' in n]
        pr = [Path(n).stem.split('__') for n in nm]
        fr = [q for q in pr if q[1] == 'front']
        print('%-5s tong %4d | front %3d | front-neutral %3d | cam xuc: %s'
              % (sp.name, len(nm), len(fr), sum(q[2] == 'neutral' for q in fr),
                 len(set(q[2] for q in fr))))
    except Exception as e:
        print('%-5s loi: %s' % (sp.name, repr(e)[:80]))
print('mat %.0f s' % (time.time() - t0))

audio.tar: 657 file | Counter({'.m4a': 657})
   audio/happy/level_1/004.m4a 83324
   audio/angry/level_3/011.m4a 99773
   audio/contempt/level_1/018.m4a 119380
   audio/happy/level_2/015.m4a 83381
   audio/disgusted/level_1/007.m4a 91897
   audio/contempt/level_2/029.m4a 128872
   audio/contempt/level_3/009.m4a 119789
   audio/angry/level_2/011.m4a 82430
front/neutral: 0
M003  tong 2668 | front 667 | front-neutral  40 | cam xuc: 8
M005  tong 2648 | front 662 | front-neutral  40 | cam xuc: 8
M007  tong 2592 | front 648 | front-neutral  40 | cam xuc: 8
M009  tong 2500 | front 625 | front-neutral  40 | cam xuc: 8
M011  tong 2668 | front 667 | front-neutral  40 | cam xuc: 8
M012  KHONG CO zip
M013  tong 2678 | front 670 | front-neutral  40 | cam xuc: 8
M019  tong 1989 | front 663 | front-neutral  40 | cam xuc: 8
M022  tong 2668 | front 667 | front-neutral  40 | cam xuc: 8
M024  tong 2656 | front 664 | front-neutral  40 | cam xuc: 8
M025  tong 2668 | front 667 | front-neutral  40 | cam xuc:

In [ ]:
import tarfile, subprocess, io
import numpy as np
from pathlib import Path
from scipy.io import wavfile

src = Path('/content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD/W028')
tf = tarfile.open(src / 'audio.tar')
names_a = [m.name for m in tf.getmembers() if m.isfile()]
EMO = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
print('Audio theo cam xuc:', {e: sum(('audio/%s/' % e) in n for n in names_a) for e in EMO})

def check(clip_name):
    _, ang, emo, L, sent = Path(clip_name).stem.split('__')
    a = 'audio/%s/level_%s/%s.m4a' % (emo, L[1:], sent)
    if a not in names_a:
        print('KHONG CO audio:', a); return
    tf.extract(a, '/tmp/mead_audio')
    wav = '/tmp/mead_audio/%s_%s_%s.wav' % (emo, L, sent)
    r = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', '/tmp/mead_audio/' + a,
                        '-ar', '16000', '-ac', '1', wav], capture_output=True, text=True)
    if r.returncode != 0:
        print('ffmpeg loi:', r.stderr[:300]); return
    sr, y = wavfile.read(wav)
    T = len(np.load(io.BytesIO(zf.read(clip_name)))['landmark_norm'])
    print('%-9s %s %s | audio %.3f s | landmark %3d frame @30fps = %.3f s | mel ~%.0f frame (hop 256)'
          % (emo, L, sent, len(y) / sr, T, T / 30, len(y) / 256))

for e in ('neutral', 'happy', 'angry'):
    c = [n for n in names if '__front__%s__' % e in n]
    if c:
        check(c[0])

Audio theo cam xuc: {'neutral': 40, 'angry': 87, 'disgusted': 87, 'fear': 86, 'happy': 90, 'sad': 90, 'surprised': 90, 'contempt': 87}
neutral   L1 001 | audio 3.712 s | landmark 112 frame @30fps = 3.733 s | mel ~232 frame (hop 256)


/tmp/ipykernel_928/1439540446.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(a, '/tmp/mead_audio')
/tmp/ipykernel_928/1439540446.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(a, '/tmp/mead_audio')


happy     L1 001 | audio 4.032 s | landmark 122 frame @30fps = 4.067 s | mel ~252 frame (hop 256)
angry     L1 001 | audio 4.011 s | landmark 121 frame @30fps = 4.033 s | mel ~251 frame (hop 256)


/tmp/ipykernel_928/1439540446.py:17: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tf.extract(a, '/tmp/mead_audio')


In [ ]:
import pickle
p = '/content/drive/MyDrive/DAT301m/Data_student/Data_maketalk_landmard_task_a/autovc_retrain_mel_test_au.pickle'
with open(p, 'rb') as f:
    au_test = pickle.load(f)

A = np.concatenate([np.asarray(x[0], dtype=np.float32) for x in au_test[:20]])
am = A.mean(0)
print('Phase A mel (20 video test): shape %s | min %.3f max %.3f mean %.3f' % (A.shape, A.min(), A.max(), A.mean()))
print('MEAD    mel (24 clip)       : shape %s | min %.3f max %.3f mean %.3f' % (M.shape, M.min(), M.max(), M.mean()))
print('tuong quan hinh dang pho: %.3f | sai lech tuyet doi TB: %.3f' % (np.corrcoef(bm, am)[0, 1], np.abs(bm - am).mean()))

Phase A mel (20 video test): shape (163431, 80) | min -0.066 max 1.048 mean 0.447
MEAD    mel (24 clip)       : shape (6175, 80) | min 0.000 max 0.982 mean 0.445
tuong quan hinh dang pho: 0.954 | sai lech tuyet doi TB: 0.025


In [ ]:
import collections
from pathlib import Path

lm = collections.defaultdict(set)
for n in names:
    p = Path(n).stem.split('__')
    if p[1] == 'front':
        lm[(p[2], p[3][1:])].add(p[4])

au = collections.defaultdict(set)
for a in names_a:
    q = a.split('/')                       # audio/<emotion>/level_<L>/<sent>.m4a
    au[(q[1], q[2].replace('level_', ''))].add(Path(q[3]).stem)

tot_lm = tot_miss = tot_extra = 0
for k in sorted(lm):
    miss = sorted(lm[k] - au[k]); extra = sorted(au[k] - lm[k])
    tot_lm += len(lm[k]); tot_miss += len(miss); tot_extra += len(extra)
    if miss or extra:
        print('%-10s L%s | landmark %2d | audio %2d | thieu audio: %s | audio thua: %s'
              % (k[0], k[1], len(lm[k]), len(au[k]), miss[:8], extra[:8]))
print('\nTong landmark front: %d | landmark khong co audio: %d | audio khong co landmark: %d'
      % (tot_lm, tot_miss, tot_extra))
print('angry L1 landmark :', sorted(lm[('angry', '1')])[:30])
print('angry L1 audio    :', sorted(au[('angry', '1')])[:30])

angry      L1 | landmark 27 | audio 27 | thieu audio: ['031', '032', '033', '034', '039', '040', '041', '042'] | audio thua: ['004', '005', '006', '011', '012', '013', '014', '019']
angry      L2 | landmark 30 | audio 30 | thieu audio: ['031', '032', '033', '034', '039', '040', '041', '042'] | audio thua: ['004', '005', '006', '011', '012', '013', '014', '019']
angry      L3 | landmark 30 | audio 30 | thieu audio: ['031', '032', '033', '034', '039', '040', '041', '042'] | audio thua: ['004', '005', '006', '011', '012', '013', '014', '019']
contempt   L1 | landmark 29 | audio 29 | thieu audio: ['033', '034', '035', '036', '041', '042', '043', '044'] | audio thua: ['004', '005', '006', '012', '013', '014', '015', '016']
contempt   L2 | landmark 29 | audio 29 | thieu audio: ['033', '034', '035', '036', '041', '042', '043', '044'] | audio thua: ['004', '005', '006', '012', '013', '014', '015', '016']
contempt   L3 | landmark 29 | audio 29 | thieu audio: ['033', '034', '035', '036', '041', 

In [ ]:
import subprocess, io, random
import numpy as np
from pathlib import Path

def dur(path):
    r = subprocess.run(['ffprobe', '-v', 'error', '-show_entries', 'format=duration',
                        '-of', 'csv=p=0', path], capture_output=True, text=True)
    return float(r.stdout.strip())

A = {}
for a in names_a:
    q = a.split('/')
    tf.extract(a, '/tmp/mead_audio', filter='data')
    A[(q[1], q[2].replace('level_', ''), Path(q[3]).stem)] = dur('/tmp/mead_audio/' + a)

Lm = {}
for n in names:
    p = Path(n).stem.split('__')
    if p[1] == 'front':
        Lm[(p[2], p[3][1:], p[4])] = len(np.load(io.BytesIO(zf.read(n)))['valid_mask']) / 30.0

random.seed(0)
d_rank, d_name, d_rand = [], [], []
for e, l in sorted({(k[0], k[1]) for k in Lm}):
    lid = sorted(k[2] for k in Lm if k[:2] == (e, l))
    aid = sorted(k[2] for k in A if k[:2] == (e, l))
    for x, y in zip(lid, aid):
        d_rank.append(abs(Lm[(e, l, x)] - A[(e, l, y)]))
    for x in lid:
        if (e, l, x) in A:
            d_name.append(abs(Lm[(e, l, x)] - A[(e, l, x)]))
        d_rand.append(abs(Lm[(e, l, x)] - A[(e, l, random.choice(aid))]))

for label, d in (('ghep theo THU TU  ', d_rank), ('ghep theo TEN     ', d_name), ('ngau nhien (moc)  ', d_rand)):
    d = np.array(d)
    print('%s | n=%3d | trung vi %.3f s | p90 %.3f s | ti le lech < 0.15 s: %.0f%%'
          % (label, len(d), np.median(d), np.percentile(d, 90), 100 * (d < 0.15).mean()))

ghep theo THU TU   | n=657 | trung vi 0.036 s | p90 0.747 s | ti le lech < 0.15 s: 80%
ghep theo TEN      | n=350 | trung vi 0.691 s | p90 2.337 s | ti le lech < 0.15 s: 26%
ngau nhien (moc)   | n=657 | trung vi 1.029 s | p90 2.563 s | ti le lech < 0.15 s: 8%


In [ ]:
groups = sorted({(k[0], k[1]) for k in Lm})
rows, PAIR = [], {}
for e, l in groups:
    lid = sorted(k[2] for k in Lm if k[:2] == (e, l))
    aid = sorted(k[2] for k in A if k[:2] == (e, l))
    for i, (x, y) in enumerate(zip(lid, aid)):
        PAIR[(e, l, x)] = y
        rows.append((e, l, i, Lm[(e, l, x)], A[(e, l, y)]))

R = np.array([(r[3], r[4]) for r in rows])
diff = np.abs(R[:, 0] - R[:, 1])
trunc = np.round(R[:, 0] * 30) >= 150            # landmark bi cat o 150 frame
ok = diff < 0.15
bad = ~ok
expl = bad & trunc & (R[:, 1] > R[:, 0])          # audio dai hon landmark bi cat
print('Tong cap: %d | lech < 0.15 s: %d | lech >= 0.15 s: %d' % (len(R), ok.sum(), bad.sum()))
print('Landmark bi cat (150 frame): %d cap' % trunc.sum())
print('Lech >= 0.15 s GIAI THICH duoc bang cat 150 frame: %d | CON LAI chua giai thich: %d'
      % (expl.sum(), (bad & ~expl).sum()))

print('\nThu dich thu tu trong tung nhom (chi tinh cap landmark khong bi cat):')
for e, l in groups:
    lid = sorted(k[2] for k in Lm if k[:2] == (e, l) and round(Lm[k] * 30) < 150)
    aid_all = sorted(k[2] for k in A if k[:2] == (e, l))
    lid_all = sorted(k[2] for k in Lm if k[:2] == (e, l))
    best, bf = 0, -1
    res = {}
    for s in (-2, -1, 0, 1, 2):
        hit = tot = 0
        for i, x in enumerate(lid_all):
            j = i + s
            if 0 <= j < len(aid_all) and round(Lm[(e, l, x)] * 30) < 150:
                tot += 1
                hit += abs(Lm[(e, l, x)] - A[(e, l, aid_all[j])]) < 0.15
        res[s] = hit / max(tot, 1)
    b = max(res, key=res.get)
    if b != 0:
        print('  %-10s L%s: dich tot nhat = %+d | ti le khop %s' % (e, l, b, {s: round(v, 2) for s, v in res.items()}))
print('(nhom nao khong in ra = dich 0 la tot nhat)')

Tong cap: 657 | lech < 0.15 s: 524 | lech >= 0.15 s: 133
Landmark bi cat (150 frame): 151 cap
Lech >= 0.15 s GIAI THICH duoc bang cat 150 frame: 133 | CON LAI chua giai thich: 0

Thu dich thu tu trong tung nhom (chi tinh cap landmark khong bi cat):
(nhom nao khong in ra = dich 0 la tot nhat)


In [ ]:
import subprocess, io
import numpy as np
from pathlib import Path

LAGS = list(range(-40, 41, 2))              # don vi: mel frame (62.5 fps), 1 frame = 16 ms
acc = {l: [] for l in LAGS}
front = [n for n in names if '__front__' in n]
used = 0
for n in front[::4]:
    _, ang, emo, L, sent = Path(n).stem.split('__')
    key = (emo, L[1:], sent)
    if key not in PAIR:
        continue
    a = 'audio/%s/level_%s/%s.m4a' % (emo, L[1:], PAIR[key])
    tf.extract(a, '/tmp/mead_audio', filter='data')
    wav = '/tmp/mead_audio/' + a.replace('/', '_') + '.wav'
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', '/tmp/mead_audio/' + a,
                    '-ar', '16000', '-ac', '1', wav], check=True)
    e_all = mead_mel(wav).mean(1)
    lm = np.load(io.BytesIO(zf.read(n)))['landmark_norm'][:, :, :2]
    mo30 = mouth_open(lm)
    T = len(mo30)
    N = min(int((T - 1) / 30.0 * 62.5) + 1, len(e_all))
    if N < 120:
        continue
    mo = np.interp(np.arange(N) / 62.5, np.arange(T) / 30.0, mo30)   # do mo mieng tren truc 62.5 fps
    e = e_all[:N]
    for l in LAGS:
        acc[l].append(corr(e[-l:], mo[:N + l]) if l < 0 else corr(e[:N - l], mo[l:]))
    used += 1

res = {l: float(np.mean(v)) for l, v in acc.items()}
best = max(res, key=res.get)
print('So clip dung:', used)
for l in LAGS:
    print('lag %+3d frame (%+5.0f ms): %.3f %s' % (l, l * 16, res[l], '<-- DINH' if l == best else ''))
print('\nDinh tai lag %+d mel-frame (%.0f ms), tuong quan %.3f   | Phase A: -10 frame, 0.515' % (best, best * 16, res[best]))

So clip dung: 163
lag -40 frame ( -640 ms): -0.014 
lag -38 frame ( -608 ms): -0.025 
lag -36 frame ( -576 ms): -0.031 
lag -34 frame ( -544 ms): -0.032 
lag -32 frame ( -512 ms): -0.026 
lag -30 frame ( -480 ms): -0.017 
lag -28 frame ( -448 ms): -0.009 
lag -26 frame ( -416 ms): -0.005 
lag -24 frame ( -384 ms): -0.007 
lag -22 frame ( -352 ms): -0.010 
lag -20 frame ( -320 ms): -0.007 
lag -18 frame ( -288 ms): 0.006 
lag -16 frame ( -256 ms): 0.038 
lag -14 frame ( -224 ms): 0.091 
lag -12 frame ( -192 ms): 0.153 
lag -10 frame ( -160 ms): 0.206 
lag  -8 frame ( -128 ms): 0.226 <-- DINH
lag  -6 frame (  -96 ms): 0.201 
lag  -4 frame (  -64 ms): 0.138 
lag  -2 frame (  -32 ms): 0.060 
lag  +0 frame (   +0 ms): -0.012 
lag  +2 frame (  +32 ms): -0.055 
lag  +4 frame (  +64 ms): -0.074 
lag  +6 frame (  +96 ms): -0.073 
lag  +8 frame ( +128 ms): -0.062 
lag +10 frame ( +160 ms): -0.053 
lag +12 frame ( +192 ms): -0.046 
lag +14 frame ( +224 ms): -0.043 
lag +16 frame ( +256 ms): -0.04

In [ ]:
import io, time, json, shutil, tarfile, zipfile, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, librosa
from scipy.io import wavfile
from scipy.signal import butter, filtfilt, get_window

ROOT = Path('/content/drive/MyDrive/DAT301m/Data_student/Data_emotion_mead_task_b/stage_b_mead_out')
MEAD = Path('/content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD')
OUT  = ROOT.parent / 'phaseB_ready'; OUT.mkdir(exist_ok=True)
EMO  = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
FPS  = 62.5

def pySTFT(x, n=1024, hop=256):
    x = np.pad(x, n // 2, mode='reflect')
    sh = x.shape[:-1] + ((x.shape[-1] - (n - hop)) // hop, n)
    st = x.strides[:-1] + (hop * x.strides[-1], x.strides[-1])
    return np.abs(np.fft.rfft(get_window('hann', n, fftbins=True) * np.lib.stride_tricks.as_strided(x, sh, st), n=n).T)
MB = librosa.filters.mel(sr=16000, n_fft=1024, fmin=90, fmax=7600, n_mels=80).T
BH, AH = butter(5, 30 / 8000, btype='high')
MINL = np.exp(-100 / 20 * np.log(10))

def decode_mel(a):
    m4a, wav = a
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', m4a, '-ar', '16000', '-ac', '1', wav], check=True)
    sr, y = wavfile.read(wav)
    x = filtfilt(BH, AH, y.astype(np.float32) / 32768.0) * 0.96
    return ((20 * np.log10(np.maximum(MINL, pySTFT(x).T @ MB)) - 16 + 100) / 100).astype(np.float32)

def process_speaker(sp, workers=4):
    t0 = time.time()
    zf = zipfile.ZipFile(ROOT / sp / ('%s_landmark_package.zip' % sp))
    tmp = Path('/tmp/mead_' + sp); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    shutil.copy(MEAD / sp / 'audio.tar', tmp / 'audio.tar')
    with tarfile.open(tmp / 'audio.tar') as tf:
        tf.extractall(tmp / 'a', filter='data')
    au = {}
    for p in (tmp / 'a' / 'audio').glob('*/level_*/*.m4a'):
        au.setdefault((p.parts[-3], p.parts[-2].replace('level_', '')), []).append(p.stem)
    lm = {}
    for n in sorted(i.filename for i in zf.infolist() if '/clips/' in i.filename and '__front__' in i.filename):
        q = Path(n).stem.split('__'); lm.setdefault((q[2], q[3][1:]), []).append((q[4], n))
    tp = np.load(io.BytesIO(zf.read('%s/statistics/derived.npz' % sp)))
    ref = tp['templates'][list(tp['template_angles']).index('front')][:, :2].reshape(136).astype(np.float32)

    jobs, bad_groups = [], []
    for k, items in lm.items():
        items, aids = sorted(items), sorted(au.get(k, []))
        if len(items) != len(aids):
            bad_groups.append((k, len(items), len(aids))); continue
        for (sid, n), aid in zip(items, aids):
            jobs.append((k, sid, n, tmp / 'a' / 'audio' / k[0] / ('level_' + k[1]) / (aid + '.m4a')))
    with ThreadPoolExecutor(workers) as ex:
        mels = list(ex.map(decode_mel, [(str(j[3]), str(tmp / ('%d.wav' % i))) for i, j in enumerate(jobs)]))

    M, R, off, emo, lvl, sent, keep, dd = [], [], [0], [], [], [], [], []
    n_inv = n_short = 0
    for (k, sid, n, _), mel in zip(jobs, mels):
        z = np.load(io.BytesIO(zf.read(n)))
        if (z['valid_mask'] == 0).any(): n_inv += 1; continue
        x = z['landmark_norm'][:, :, :2].astype(np.float32).reshape(-1, 136)
        T = len(x); N = min(int((T - 1) / 30 * FPS) + 1, len(mel))
        if N < 40: n_short += 1; continue
        t = np.arange(N) / FPS
        l62 = np.stack([np.interp(t, np.arange(T) / 30.0, x[:, j]) for j in range(136)], 1)
        if T < 150: dd.append(abs(len(mel) / FPS - T / 30.0))
        M.append(mel[:N]); R.append(l62 - ref)
        off.append(off[-1] + N); emo.append(EMO.index(k[0])); lvl.append(int(k[1])); sent.append(int(sid)); keep.append(Path(n).stem)
    f = '/tmp/%s.npz' % sp
    np.savez(f, mel=np.concatenate(M).astype(np.float16), lm_res=np.concatenate(R).astype(np.float16),
             offsets=np.array(off, dtype=np.int64), emo=np.array(emo, dtype=np.int8), lvl=np.array(lvl, dtype=np.int8),
             sent=np.array(sent, dtype=np.int16), ref=ref, names=np.array(keep))
    shutil.copy(f, OUT / (sp + '.npz'))
    agree = float((np.array(dd) < 0.15).mean()) if dd else 0.0
    st = dict(speaker=sp, clips=len(keep), frames=int(off[-1]), invalid_dropped=n_inv, short_dropped=n_short,
              bad_groups=[(g, a, b) for g, a, b in bad_groups], duration_agree=round(agree, 3),
              ok=(agree >= 0.9 and not bad_groups), seconds=round(time.time() - t0, 1))
    (OUT / (sp + '.json')).write_text(json.dumps(st, indent=1))
    shutil.rmtree(tmp, ignore_errors=True)
    return st

st = process_speaker('W028')
print(st)
print('Dung luong file: %.0f MB' % ((OUT / 'W028.npz').stat().st_size / 1e6))

{'speaker': 'W028', 'clips': 656, 'frames': 156292, 'invalid_dropped': 1, 'short_dropped': 0, 'bad_groups': [], 'duration_agree': 1.0, 'ok': True, 'seconds': 121.8}
Dung luong file: 68 MB


In [ ]:
def smooth(x, k):
    ker = np.ones(k) / k
    return np.stack([np.convolve(x[:, j], ker, mode='same') for j in range(x.shape[1])], 1)

for k in (3, 5):
    print('\nCua so lam muot %d frame (%.0f ms)' % (k, k * 1000 / 30))
    res = {r: [] for r in REG}; mot = {r: [] for r in REG}
    for n in names:
        x = np.load(io.BytesIO(zf.read(n)))['landmark_norm'][:, :, :2].astype(np.float64)
        if len(x) < 30: continue
        f = x.reshape(len(x), 136)
        e = np.linalg.norm((f - smooth(f, k))[k:-k].reshape(-1, 68, 2), axis=2)
        c = np.linalg.norm((x - x.mean(0)), axis=2)
        for r, idx in REG.items():
            idx = list(idx); res[r].append(e[:, idx].mean()); mot[r].append(c[:, idx].mean())
    for r in REG:
        a, b = np.mean(res[r]), np.mean(mot[r])
        print('  %-6s rung %.5f | chuyen dong %.5f | ti le %.1f%%' % (r, a, b, 100 * a / b))


Cua so lam muot 3 frame (100 ms)
  ham    rung 0.00267 | chuyen dong 0.01474 | ti le 18.1%
  may    rung 0.00143 | chuyen dong 0.00483 | ti le 29.5%
  mat    rung 0.00075 | chuyen dong 0.00250 | ti le 29.9%
  mieng  rung 0.00351 | chuyen dong 0.02570 | ti le 13.7%

Cua so lam muot 5 frame (167 ms)
  ham    rung 0.00420 | chuyen dong 0.01474 | ti le 28.5%
  may    rung 0.00192 | chuyen dong 0.00483 | ti le 39.7%
  mat    rung 0.00099 | chuyen dong 0.00250 | ti le 39.5%
  mieng  rung 0.00667 | chuyen dong 0.02570 | ti le 25.9%


In [ ]:
import io, time, json, shutil, tarfile, zipfile, subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np, librosa
from scipy.io import wavfile
from scipy.signal import butter, filtfilt, get_window, savgol_filter

ROOT = Path('/content/drive/MyDrive/DAT301m/Data_student/Data_emotion_mead_task_b/stage_b_mead_out')
MEAD = Path('/content/drive/.shortcut-targets-by-id/1GwXP-KpWOxOenOxITTsURJZQ_1pkd4-j/MEAD')
OUT  = ROOT.parent / 'phaseB_ready'; OUT.mkdir(exist_ok=True)
EMO  = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
FPS  = 62.5
SG_WIN, SG_ORD = 5, 2          # lam muot o 30 fps: 5 frame = 167 ms, bac 2

def pySTFT(x, n=1024, hop=256):
    x = np.pad(x, n // 2, mode='reflect')
    sh = x.shape[:-1] + ((x.shape[-1] - (n - hop)) // hop, n)
    st = x.strides[:-1] + (hop * x.strides[-1], x.strides[-1])
    return np.abs(np.fft.rfft(get_window('hann', n, fftbins=True) * np.lib.stride_tricks.as_strided(x, sh, st), n=n).T)
MB = librosa.filters.mel(sr=16000, n_fft=1024, fmin=90, fmax=7600, n_mels=80).T
BH, AH = butter(5, 30 / 8000, btype='high')
MINL = np.exp(-100 / 20 * np.log(10))

def decode_mel(a):
    m4a, wav = a
    subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', m4a, '-ar', '16000', '-ac', '1', wav], check=True)
    sr, y = wavfile.read(wav)
    x = filtfilt(BH, AH, y.astype(np.float32) / 32768.0) * 0.96
    return ((20 * np.log10(np.maximum(MINL, pySTFT(x).T @ MB)) - 16 + 100) / 100).astype(np.float32)

def to62(x, N):
    t = np.arange(N) / FPS
    return np.stack([np.interp(t, np.arange(len(x)) / 30.0, x[:, j]) for j in range(x.shape[1])], 1)

def process_speaker(sp, workers=4):
    t0 = time.time()
    zf = zipfile.ZipFile(ROOT / sp / ('%s_landmark_package.zip' % sp))
    tmp = Path('/tmp/mead_' + sp); shutil.rmtree(tmp, ignore_errors=True); tmp.mkdir(parents=True)
    shutil.copy(MEAD / sp / 'audio.tar', tmp / 'audio.tar')
    with tarfile.open(tmp / 'audio.tar') as tf:
        tf.extractall(tmp / 'a', filter='data')
    au = {}
    for p in (tmp / 'a' / 'audio').glob('*/level_*/*.m4a'):
        au.setdefault((p.parts[-3], p.parts[-2].replace('level_', '')), []).append(p.stem)
    lm = {}
    for n in sorted(i.filename for i in zf.infolist() if '/clips/' in i.filename and '__front__' in i.filename):
        q = Path(n).stem.split('__'); lm.setdefault((q[2], q[3][1:]), []).append((q[4], n))
    tp = np.load(io.BytesIO(zf.read('%s/statistics/derived.npz' % sp)))
    ref = tp['templates'][list(tp['template_angles']).index('front')][:, :2].reshape(136).astype(np.float32)

    jobs, bad_groups = [], []
    for k, items in lm.items():
        items, aids = sorted(items), sorted(au.get(k, []))
        if len(items) != len(aids):
            bad_groups.append((k, len(items), len(aids))); continue
        for (sid, n), aid in zip(items, aids):
            jobs.append((k, sid, n, tmp / 'a' / 'audio' / k[0] / ('level_' + k[1]) / (aid + '.m4a')))
    with ThreadPoolExecutor(workers) as ex:
        mels = list(ex.map(decode_mel, [(str(j[3]), str(tmp / ('%d.wav' % i))) for i, j in enumerate(jobs)]))

    M, R, RR, off, emo, lvl, sent, keep, dd = [], [], [], [0], [], [], [], [], []
    n_inv = n_short = 0
    for (k, sid, n, _), mel in zip(jobs, mels):
        z = np.load(io.BytesIO(zf.read(n)))
        if (z['valid_mask'] == 0).any(): n_inv += 1; continue
        x = z['landmark_norm'][:, :, :2].astype(np.float32).reshape(-1, 136)
        T = len(x); N = min(int((T - 1) / 30 * FPS) + 1, len(mel))
        if N < 40 or T < SG_WIN: n_short += 1; continue
        xs = savgol_filter(x, SG_WIN, SG_ORD, axis=0)
        if T < 150: dd.append(abs(len(mel) / FPS - T / 30.0))
        M.append(mel[:N]); R.append(to62(xs, N) - ref); RR.append(to62(x, N) - ref)
        off.append(off[-1] + N); emo.append(EMO.index(k[0])); lvl.append(int(k[1])); sent.append(int(sid)); keep.append(Path(n).stem)
    f = '/tmp/%s.npz' % sp
    np.savez(f, mel=np.concatenate(M).astype(np.float16),
             lm_res=np.concatenate(R).astype(np.float16), lm_res_raw=np.concatenate(RR).astype(np.float16),
             offsets=np.array(off, dtype=np.int64), emo=np.array(emo, dtype=np.int8), lvl=np.array(lvl, dtype=np.int8),
             sent=np.array(sent, dtype=np.int16), ref=ref, names=np.array(keep),
             smooth=np.array([SG_WIN, SG_ORD]))
    shutil.copy(f, OUT / (sp + '.npz'))
    agree = float((np.array(dd) < 0.15).mean()) if dd else 0.0
    st = dict(speaker=sp, clips=len(keep), frames=int(off[-1]), invalid_dropped=n_inv, short_dropped=n_short,
              bad_groups=[(g, a, b) for g, a, b in bad_groups], duration_agree=round(agree, 3),
              ok=(agree >= 0.9 and not bad_groups), seconds=round(time.time() - t0, 1))
    (OUT / (sp + '.json')).write_text(json.dumps(st, indent=1))
    shutil.rmtree(tmp, ignore_errors=True)
    return st

st = process_speaker('W028')
print(st)
print('Dung luong file: %.0f MB' % ((OUT / 'W028.npz').stat().st_size / 1e6))

{'speaker': 'W028', 'clips': 656, 'frames': 156292, 'invalid_dropped': 1, 'short_dropped': 0, 'bad_groups': [], 'duration_agree': 1.0, 'ok': True, 'seconds': 117.0}
Dung luong file: 110 MB


In [ ]:
import json, time, traceback

SKIP = {'M012', 'M029', 'M042', 'W021', 'W017'}
speakers = sorted(p.name for p in ROOT.iterdir()
                  if p.is_dir() and p.name not in SKIP and (p / ('%s_landmark_package.zip' % p.name)).exists())
print('So speaker se xu ly:', len(speakers))

t_all = time.time()
for i, sp in enumerate(speakers, 1):
    js = OUT / (sp + '.json')
    if js.exists() and json.loads(js.read_text()).get('ok') and (OUT / (sp + '.npz')).exists():
        print('[%2d/%d] %s da xong, bo qua' % (i, len(speakers), sp)); continue
    try:
        st = process_speaker(sp)
        print('[%2d/%d] %s | clip %d | ok %s | khop %.2f | lech nhom %d | %.0f s'
              % (i, len(speakers), sp, st['clips'], st['ok'], st['duration_agree'], len(st['bad_groups']), st['seconds']))
    except Exception as e:
        (OUT / (sp + '.json')).write_text(json.dumps(dict(speaker=sp, ok=False, error=repr(e)[:500])))
        print('[%2d/%d] %s LOI: %s' % (i, len(speakers), sp, repr(e)[:200]))
print('\nTong thoi gian: %.0f phut' % ((time.time() - t_all) / 60))

rows = [json.loads((OUT / (sp + '.json')).read_text()) for sp in speakers if (OUT / (sp + '.json')).exists()]
good = [r for r in rows if r.get('ok')]
print('Dat: %d | Khong dat: %d' % (len(good), len(rows) - len(good)))
for r in rows:
    if not r.get('ok'):
        print('  ', r['speaker'], r.get('error') or ('khop %s | lech nhom %s' % (r.get('duration_agree'), r.get('bad_groups'))))
print('Tong clip: %d | tong frame: %d' % (sum(r['clips'] for r in good), sum(r['frames'] for r in good)))

So speaker se xu ly: 41
[ 1/41] M003 | clip 667 | ok True | khop 1.00 | lech nhom 0 | 103 s
[ 2/41] M005 | clip 662 | ok True | khop 1.00 | lech nhom 0 | 116 s
[ 3/41] M007 | clip 648 | ok True | khop 1.00 | lech nhom 0 | 102 s
[ 4/41] M009 | clip 625 | ok True | khop 1.00 | lech nhom 0 | 97 s
[ 5/41] M011 | clip 667 | ok True | khop 1.00 | lech nhom 0 | 106 s
[ 6/41] M013 | clip 670 | ok True | khop 1.00 | lech nhom 0 | 110 s
[ 7/41] M019 | clip 663 | ok True | khop 1.00 | lech nhom 0 | 114 s
[ 8/41] M022 | clip 667 | ok True | khop 1.00 | lech nhom 0 | 111 s
[ 9/41] M024 | clip 664 | ok True | khop 1.00 | lech nhom 0 | 111 s
[10/41] M025 | clip 667 | ok True | khop 1.00 | lech nhom 0 | 109 s
[11/41] M026 | clip 667 | ok True | khop 1.00 | lech nhom 0 | 107 s
[12/41] M028 | clip 663 | ok True | khop 1.00 | lech nhom 0 | 110 s
[13/41] M030 | clip 600 | ok False | khop 1.00 | lech nhom 2 | 104 s
[14/41] M031 | clip 662 | ok True | khop 1.00 | lech nhom 0 | 114 s
[15/41] M032 | clip 667 

In [ ]:
BAD_AUDIO = []
def decode_mel(a):
    m4a, wav = a
    r = subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', m4a, '-ar', '16000', '-ac', '1', wav],
                       capture_output=True, text=True)
    if r.returncode != 0:
        BAD_AUDIO.append(m4a)
        return np.zeros((0, 80), np.float32)          # mel rong -> clip bi loai o buoc kiem tra do dai
    sr, y = wavfile.read(wav)
    x = filtfilt(BH, AH, y.astype(np.float32) / 32768.0) * 0.96
    return ((20 * np.log10(np.maximum(MINL, pySTFT(x).T @ MB)) - 16 + 100) / 100).astype(np.float32)

st = process_speaker('M041')
print(st)
print('Audio khong giai ma duoc:', BAD_AUDIO)

{'speaker': 'M041', 'clips': 663, 'frames': 149827, 'invalid_dropped': 1, 'short_dropped': 3, 'bad_groups': [], 'duration_agree': 1.0, 'ok': True, 'seconds': 113.6}
Audio khong giai ma duoc: ['/tmp/mead_M041/a/audio/sad/level_2/011.m4a', '/tmp/mead_M041/a/audio/sad/level_2/012.m4a', '/tmp/mead_M041/a/audio/sad/level_2/013.m4a', '/tmp/mead_M041/a/audio/sad/level_2/014.m4a']


In [ ]:
import json
import numpy as np

EMO = ['neutral', 'angry', 'disgusted', 'fear', 'happy', 'sad', 'surprised', 'contempt']
REG = {'ham': range(0, 17), 'may': range(17, 27), 'mui': range(27, 36), 'mat': range(36, 48), 'mieng': range(48, 68)}

# ---- chia speaker ----
usable = []
for js in sorted(OUT.glob('*.json')):
    if js.name == 'split.json': continue
    r = json.loads(js.read_text())
    if 'error' not in r and r.get('duration_agree', 0) >= 0.9 and (OUT / (r['speaker'] + '.npz')).exists():
        usable.append(r['speaker'])
partial = {'M030', 'W014'}
rng = np.random.default_rng(0)
def pick(pool, k):
    return [str(s) for s in rng.choice(sorted(s for s in pool if s not in partial), k, replace=False)]
men, women = [s for s in usable if s[0] == 'M'], [s for s in usable if s[0] == 'W']
test = pick(men, 2) + pick(women, 2)
val = pick([s for s in men if s not in test], 2) + pick([s for s in women if s not in test], 2)
train = sorted(s for s in usable if s not in test + val)
split = dict(train=train, val=sorted(val), test=sorted(test), seed=0)
(OUT / 'split.json').write_text(json.dumps(split, indent=1))
print('Dung duoc: %d | train %d | val %d %s | test %d %s' % (len(usable), len(train), len(val), split['val'], len(test), split['test']))

def load(sp):
    z = np.load(OUT / (sp + '.npz'))
    return z['lm_res_raw'].astype(np.float32), z['offsets'], z['emo'], z['lvl']

# ---- do lech tinh trung binh theo (cam xuc, muc), CHI tu speaker train ----
S, C = {}, {}
for sp in train:
    x, off, emo, lvl = load(sp)
    for i in range(len(off) - 1):
        k = (int(emo[i]), int(lvl[i]))
        S[k] = S.get(k, 0) + x[off[i]:off[i + 1]].mean(0); C[k] = C.get(k, 0) + 1
OFF = {k: S[k] / C[k] for k in S}
print('Kiem tra: do lech TB cua neutral (phai nho): %.4f | cua happy L3: %.4f'
      % (np.abs(OFF[(0, 1)]).mean(), np.abs(OFF[(4, 3)]).mean()))

# ---- danh gia tren speaker test ----
CAND = ['1 dung yen (mat trung tinh)', '2 + lech TB (cam xuc, muc) tu speaker train', '3 tran: TB cua chinh clip']
acc, per_sp = {}, {}
for sp in test:
    x, off, emo, lvl = load(sp)
    for i in range(len(off) - 1):
        g = x[off[i]:off[i + 1]]; k = (int(emo[i]), int(lvl[i]))
        grp = 'neutral' if k[0] == 0 else 'emotional'
        for name, p in zip(CAND, (np.zeros(136, np.float32), OFF[k], g.mean(0))):
            e = np.linalg.norm((g - p).reshape(len(g), 68, 2), axis=2)
            a = acc.setdefault((grp, name), [np.zeros(68), 0]); a[0] += e.sum(0); a[1] += len(g)
            if grp == 'emotional':
                b = per_sp.setdefault((sp, name), [0.0, 0]); b[0] += e.mean(1).sum(); b[1] += len(g)

for grp in ('neutral', 'emotional'):
    base = acc[(grp, CAND[0])]; b = base[0] / base[1]
    print('\n== clip %s (4 speaker test) ==' % grp)
    for name in CAND:
        m = acc[(grp, name)][0] / acc[(grp, name)][1]
        print('  %-46s toan mat %.4f (%+5.1f%%) | mieng %.4f (%+5.1f%%)'
              % (name, m.mean(), 100 * (1 - m.mean() / b.mean()), m[48:].mean(), 100 * (1 - m[48:].mean() / b[48:].mean())))
    if grp == 'emotional':
        m2 = acc[(grp, CAND[1])][0] / acc[(grp, CAND[1])][1]
        print('  Theo vung (cach 2 so voi dung yen):',
              ', '.join('%s %+.1f%%' % (r, 100 * (1 - m2[list(i)].mean() / b[list(i)].mean())) for r, i in REG.items()))

print('\nTung speaker test (clip cam xuc, toan mat, cach 2 so voi dung yen):')
for sp in test:
    a, c = per_sp[(sp, CAND[0])], per_sp[(sp, CAND[1])]
    print('  %s: %+.1f%%' % (sp, 100 * (1 - (c[0] / c[1]) / (a[0] / a[1]))))

Dung duoc: 41 | train 33 | val 4 ['M003', 'M005', 'W033', 'W036'] | test 4 ['M032', 'M037', 'W018', 'W019']
Kiem tra: do lech TB cua neutral (phai nho): 0.0001 | cua happy L3: 0.0133

== clip neutral (4 speaker test) ==
  1 dung yen (mat trung tinh)                    toan mat 0.0221 ( +0.0%) | mieng 0.0345 ( +0.0%)
  2 + lech TB (cam xuc, muc) tu speaker train    toan mat 0.0221 ( -0.0%) | mieng 0.0345 ( -0.0%)
  3 tran: TB cua chinh clip                      toan mat 0.0159 (+28.1%) | mieng 0.0282 (+18.3%)

== clip emotional (4 speaker test) ==
  1 dung yen (mat trung tinh)                    toan mat 0.0488 ( +0.0%) | mieng 0.0677 ( +0.0%)
  2 + lech TB (cam xuc, muc) tu speaker train    toan mat 0.0435 (+10.7%) | mieng 0.0605 (+10.8%)
  3 tran: TB cua chinh clip                      toan mat 0.0179 (+63.3%) | mieng 0.0334 (+50.6%)
  Theo vung (cach 2 so voi dung yen): ham +5.4%, may +25.0%, mui +8.8%, mat +10.8%, mieng +10.8%

Tung speaker test (clip cam xuc, toan mat, cach 2 so vo